In [17]:
import cv2
import numpy as np
import mediapipe as mp
import joblib
from collections import deque
import math


In [18]:
model = joblib.load('banminton_pose_model.pkl')

In [19]:
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils
pose = mp_pose.Pose()

In [20]:
class_name = ['Backhand', 'Forehand_Defense', 'Smash', 'Stance']

In [21]:
history = deque(maxlen=10)

In [22]:

def extract_features(lmlist):


    nose = lmlist[0][1:]
    right_shoulder = lmlist[12][1:]
    right_wrist = lmlist[16][1:]
    right_hip = lmlist[24][1:]

    right_hand_up = 1 if right_wrist[1] < right_shoulder[1] else 0
    right_cross_body = 1 if right_wrist[0] < nose[0] else 0
    lean_right = 1 if nose[0] >  right_hip[0]  else 0
    row = [
        right_hand_up,
        right_cross_body,
        lean_right,
    ]

    return row

In [23]:
video = cv2.VideoCapture(0)

video.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
video.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

required_landmarks = [
    0,   # nose
    12,   # shoulders
    16,   # wrists
    24,   # hips
]

while True:

    ret, img = video.read()

    if not ret:
        break

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    result = pose.process(rgb)

    landmark_visible = True

    if result.pose_landmarks:

        mp_draw.draw_landmarks(
            img,
            result.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

        lmlist = []

        for idx, lm in enumerate(result.pose_landmarks.landmark):
            lmlist.append([idx, lm.x, lm.y, lm.z, lm.visibility])

        for idx in required_landmarks:

            visibility = lmlist[idx][4]

            # visibility threshold
            if visibility < 0.60:
                landmark_visible = False
                break


        if not landmark_visible:

            cv2.putText(
                img,
                "Landmark Not Visible",
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                3
            )

        else:

            try:

                row = extract_features(lmlist)


                row = np.array(row).reshape(1, -1)


                prediction = model.predict_proba(row)[0]

                pred_class = np.argmax(prediction)

                history.append(pred_class)

                final_pred = max(set(history), key=history.count)

                max_prob = np.max(prediction)


                label = (
                    class_name[final_pred]
                    if max_prob >= 0.70
                    else "Uncertain"
                )


                cv2.putText(
                    img,
                    label,
                    (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    2
                )


                bar_x = 170
                bar_width_max = 200
                start_y = 80

                for i, val in enumerate(prediction):

                    y = start_y + i * 35

                    percent = int(val * 100)

                    bar_width = int(val * bar_width_max)

                    cv2.putText(
                        img,
                        class_name[i],
                        (10, y),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255, 255, 255),
                        1
                    )

                    cv2.rectangle(
                        img,
                        (bar_x, y - 10),
                        (bar_x + bar_width, y + 10),
                        (0, 255, 0),
                        -1
                    )

                    cv2.putText(
                        img,
                        f"{percent}%",
                        (bar_x + bar_width + 5, y),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (255, 255, 255),
                        1
                    )

            except Exception as e:
                print("Error:", e)

    else:

        cv2.putText(
            img,
            "No Pose Detected",
            (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            3
        )

    cv2.imshow("Pose Prediction", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()

c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.w